# 05 — Genie Space + AI Skill + Demo Guide

**What this does:** Creates the natural language interface for business users and registers a reusable AI Skill.

| Component | Who Uses It | Rubric Criterion |
|-----------|-------------|------------------|
| Genie Space | Business analysts, supervisors | Criterion 5 (UX) |
| AI Skill (UC Function) | Any team in the organization | Criterion 6 (Innovation) |
| Demo Guide | Your pod for Day 2 presentation | All criteria |

---
### Rubric Targets
> *Criterion 5: "Polished UI; fluid natural-language querying returns clear, trustworthy answers"*  
> *Criterion 6: "Novel technique or reusable asset (e.g., a shared AI skill)"*

---
*Prerequisites: Run 00 → 01 → 02 → **03** in order. Notebook 03 must run first — it validates AI scoring quality and writes results to `eval_quality_metrics`. The cell below will block this notebook if the pipeline and judge differ by more than 1 point on average, preventing mis-calibrated scores from being exposed through Genie.*

In [0]:
# Quality gate: block Genie Space setup if the scorer and judge are materially mis-calibrated.
# Architecture: Claude Sonnet 4 (scorer) + Gemini 2.5 Pro (judge)
# Genie would expose unvalidated scores directly to business analysts — higher trust bar.
# Run notebook 03 (03_LLM_Judge_Evals) first to populate eval_quality_metrics.
from pyspark.errors import AnalysisException

try:
    metrics = spark.sql("""
      SELECT pct_within_1_point, mean_absolute_error, quality_gate,
             avg_pipeline_score, avg_judge_score, evaluated_at
      FROM mmt_aws_usw2_catalog.contact_calls.eval_quality_metrics
      ORDER BY evaluated_at DESC
      LIMIT 1
    """)
    rows = metrics.collect()
except AnalysisException:
    rows = []

if not rows:
    raise Exception(
        "QUALITY GATE FAILED: No eval metrics found.\n"
        "Run notebook 03 (03_LLM_Judge_Evals) first to validate AI scoring quality.\n"
        "Genie must not expose unvalidated scores to business analysts."
    )

r = rows[0]
if r["quality_gate"] == "FAIL":
    inflation = r["avg_pipeline_score"] - r["avg_judge_score"]
    raise Exception(
        f"QUALITY GATE FAILED: Pipeline and judge differ by more than 1.0 point on average.\n"
        f"  Pipeline avg: {r['avg_pipeline_score']}, Judge avg: {r['avg_judge_score']}\n"
        f"  Calibration gap: {inflation:+.2f}\n"
        f"  Do not expose mis-calibrated scores through Genie. Re-run notebook 03 after investigating."
    )

print("Quality gate PASSED")
print(f"  Pipeline avg score : {r['avg_pipeline_score']}")
print(f"  Judge avg score    : {r['avg_judge_score']} (Gemini 2.5 Pro, cross-provider check)")
print(f"  Quality gate       : {r['quality_gate']}")
print(f"  Eval validated at  : {r['evaluated_at']}")
print()
print("Scores are validated. Proceeding to Genie Space setup.")

Quality gate PASSED
  Pipeline avg score : 2.72
  Judge avg score    : 2.1 (Gemini 2.5 Pro, cross-provider check)
  Quality gate       : PASS
  Eval validated at  : 2026-06-26 07:39:49.397467

Scores are validated. Proceeding to Genie Space setup.


In [0]:
%sql
-- This UC function can be reused by ANY team in the organization for their own use cases
-- (patient feedback, employee surveys, social media monitoring, etc.)
CREATE OR REPLACE FUNCTION mmt_aws_usw2_catalog.contact_calls.ai_skill_sentiment_analysis(text STRING)
RETURNS STRING
COMMENT 'Reusable AI Skill: Analyzes sentiment of any text. Returns JSON with label, confidence, and emotional indicators. Portable across all use cases.'
RETURN (
  SELECT ai_query(
    'databricks-claude-sonnet-4-6',
    CONCAT(
      'Analyze sentiment. Return JSON: {"sentiment": "Positive"|"Negative"|"Neutral"|"Mixed", "confidence": 0.0-1.0, "summary": "one sentence"}\n\nText: ', text
    )
  )
)

In [0]:
%sql
-- Test: anyone in the org can now call this function
SELECT mmt_aws_usw2_catalog.contact_calls.ai_skill_sentiment_analysis(
  'The agent was incredibly helpful and resolved my billing issue in under 2 minutes. Best experience ever!'
) AS sentiment_result

sentiment_result
"```json { ""sentiment"": ""Positive"", ""confidence"": 0.98, ""summary"": ""Customer had an outstanding experience with fast, effective billing support from the agent."" } ```"


## Create Your Genie Space (PM Action!)

**Steps:**
1. In the left sidebar, click **Genie**
2. Click **New Space**
3. Name: **"Contact Center QA Insights"**
4. Add table: `mmt_aws_usw2_catalog.contact_calls.gold_scorecard`
5. Add these **instructions** to the Genie Space:

```
This space contains post-call quality evaluation data for a contact center.

Key metrics:
- overall_qa_score: Weighted average of all criteria (1.0-5.0)
- Individual scores (1-5): greeting_score, identity_verification_score, empathy_score, commitment_score, branding_score, compliance_score, resolution_score, further_assistance_score, closing_score, customer_service_score
- greeting_adherence / closing_adherence: Script similarity scores (0.0-1.0)
- requires_human_review: Boolean flag for outlier calls needing supervisor attention

Dimensions:
- agent_name: Contact center agent
- queue: Department (Appointments, Billing, Nurse Advice, Referrals, Pharmacy, Insurance Verification, Medical Records)
- call_category: AI-classified call type (Billing Dispute, Appointment Scheduling, Clinical Triage, Complaint, General Inquiry, Insurance Question, Prescription Refill)
- disposition: AI-classified outcome (escalate_to_supervisor, routine, coaching_opportunity)
- protocol_adherence: Compliance classification (compliant, partially_compliant, non_compliant)
- sentiment: Customer sentiment (positive/negative/neutral/mixed)
- direction: inbound/outbound
- division: Hospital/clinic division
```

**Example questions to try:**
* "Who is the best performing agent?"
* "Which queue has the lowest empathy scores?"
* "Show me calls that need human review"
* "What's the average score by department?"
* "Which agents need coaching on greeting?"

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Configuration
SPACE_TITLE = "Contact Center QA Insights"
TABLE = "mmt_aws_usw2_catalog.contact_calls.gold_scorecard"

# General instructions for the Genie Space (from markdown above)
INSTRUCTIONS = """
This space contains post-call quality evaluation data for a contact center.

Key metrics:
- overall_qa_score: Weighted average of all criteria (1.0-5.0)
- Individual scores (1-5): greeting_score, identity_verification_score, empathy_score, commitment_score, branding_score, compliance_score, resolution_score, further_assistance_score, closing_score, customer_service_score
- greeting_adherence / closing_adherence: Script similarity scores (0.0-1.0)
- requires_human_review: Boolean flag for outlier calls needing supervisor attention

Dimensions:
- agent_name: Contact center agent
- queue: Department (Appointments, Billing, Nurse Advice, Referrals, Pharmacy, Insurance Verification, Medical Records)
- call_category: AI-classified call type (Billing Dispute, Appointment Scheduling, Clinical Triage, Complaint, General Inquiry, Insurance Question, Prescription Refill)
- disposition: AI-classified outcome (escalate_to_supervisor, routine, coaching_opportunity)
- protocol_adherence: Compliance classification (compliant, partially_compliant, non_compliant)
- sentiment: Customer sentiment (positive/negative/neutral/mixed)
- direction: inbound/outbound
- division: Hospital/clinic division
"""

# Create the Genie Space via REST API (idempotent — reuses existing space if found)
import json as _json

# Check if a space with this title already exists
existing_spaces = w.api_client.do("GET", "/api/2.0/genie/spaces")
matching = [s for s in existing_spaces.get("spaces", []) if s["title"] == SPACE_TITLE]

if matching:
    space_id = matching[0]["space_id"]
    print(f"Genie Space already exists — reusing it.")
else:
    # Get an available SQL warehouse
    warehouses = [wh for wh in w.warehouses.list()]
    running = [wh for wh in warehouses if str(wh.state) == "State.RUNNING"]
    warehouse_id = running[0].id if running else warehouses[0].id

    serialized_space = _json.dumps({
        "version": 2,
        "data_sources": {
            "tables": [{"identifier": TABLE}]
        }
    })

    space_payload = {
        "title": SPACE_TITLE,
        "description": "Natural language interface for contact center QA supervisors and analysts.",
        "serialized_space": serialized_space,
        "warehouse_id": warehouse_id,
    }

    result = w.api_client.do("POST", "/api/2.0/genie/spaces", body=space_payload)
    space_id = result["space_id"]
    print(f"Genie Space created!")

GENIE_SPACE_ID = space_id

print(f"  Title:    {SPACE_TITLE}")
print(f"  Space ID: {space_id}")
print(f"  URL:      https://{spark.conf.get('spark.databricks.workspaceUrl')}/genie/rooms/{space_id}")
print()
print("Adding instructions via REST API...")

# Update with instructions
update_resp = w.api_client.do("PATCH", f"/api/2.0/genie/spaces/{space_id}", body={"description": INSTRUCTIONS})

if update_resp is not None:
    print("Instructions added successfully.")
else:
    print("Note: Could not add instructions via API. Add manually in the Genie UI.")

print(f"\nGENIE_SPACE_ID = \"{space_id}\"")

Genie Space already exists — reusing it.
  Title:    Contact Center QA Insights
  Space ID: 01f16edf130d17cdb84846cfc6fa735f
  URL:      https://fevm-mmt-aws-usw2.cloud.databricks.com/genie/rooms/01f16edf130d17cdb84846cfc6fa735f

Adding instructions via REST API...
Instructions added successfully.

GENIE_SPACE_ID = "01f16edf130d17cdb84846cfc6fa735f"


In [0]:
# Genie Space created via API — uses GENIE_SPACE_ID from cell above
GENIE_SPACE_URL = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/genie/rooms/{GENIE_SPACE_ID}"

print(f"Genie Space: Contact Center QA Insights")
print(f"Space ID:    {GENIE_SPACE_ID}")
print(f"URL:         {GENIE_SPACE_URL}")
print(f"Table:       mmt_aws_usw2_catalog.contact_calls.gold_scorecard")
print()
print("Next steps:")
print("  1. Open the Genie Space and add the General Instructions from the markdown above")
print("  2. Add sample questions to guide users")
print("  3. Test with: 'Who is the best performing agent?'")

Genie Space: Contact Center QA Insights
Space ID:    01f16edf130d17cdb84846cfc6fa735f
URL:         https://fevm-mmt-aws-usw2.cloud.databricks.com/genie/rooms/01f16edf130d17cdb84846cfc6fa735f
Table:       mmt_aws_usw2_catalog.contact_calls.gold_scorecard

Next steps:
  1. Open the Genie Space and add the General Instructions from the markdown above
  2. Add sample questions to guide users
  3. Test with: 'Who is the best performing agent?'


## Test Genie Space: UI Walkthrough

**Open the space:** [Contact Center QA Insights](https://fevm-mmt-aws-usw2.cloud.databricks.com/genie/rooms/01f16edf130d17cdb84846cfc6fa735f)

### Demo Script (60 seconds)

1. **Open Genie** — Click the link above or navigate via the left sidebar → Genie
2. **Ask a simple question:**
   > "Who is the best performing agent?"
3. **Show the generated SQL** — click the SQL tab to prove transparency
4. **Ask a follow-up** (same conversation):
   > "Break that down by queue"
5. **Ask a coaching question:**
   > "Which agents need coaching on empathy?"
6. **Show a filter question:**
   > "Show me calls that need human review in Billing"

### What to Highlight for Judges

| Feature | What to Say |
|---------|-------------|
| Natural language | "No SQL required — supervisors ask in plain English" |
| Generated SQL visible | "Full transparency — users can verify what Genie did" |
| Follow-up context | "Genie remembers context within a conversation" |
| Table grounding | "Answers come only from validated gold_scorecard data" |
| Quality gate | "Scores are only exposed after passing 80% agreement validation" |

### Troubleshooting

* **"I don't have enough information"** → Check that General Instructions were added to the space
* **Wrong column referenced** → Verify table is `mmt_aws_usw2_catalog.contact_calls.gold_scorecard`
* **Warehouse error** → Ensure the SQL warehouse is running (Serverless Starter Warehouse)

In [0]:
import time

# Test the Genie Space with a natural language question
question = "Which agents need coaching on empathy?"

result = w.api_client.do("POST", f"/api/2.0/genie/spaces/{GENIE_SPACE_ID}/start-conversation", body={
    "content": question
})

conversation_id = result["conversation_id"]
message_id = result["message_id"]

# Poll for completion
for _ in range(30):
    msg = w.api_client.do("GET", f"/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}")
    status = msg.get("status")
    if status in ("COMPLETED", "FAILED"):
        break
    time.sleep(2)

print(f"Question: {question}")
print(f"Status:   {status}")
print()

if "attachments" in msg:
    for att in msg["attachments"]:
        if "query" in att:
            print(f"Generated SQL:\n{att['query'].get('query', 'N/A')}\n")
        if "text" in att:
            print(f"Answer:\n{att['text'].get('content', '')}")

Question: Which agents need coaching on empathy?
Status:   COMPLETED

Generated SQL:
SELECT gold_scorecard.agent_name
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
WHERE gold_scorecard.empathy_score IS NOT NULL AND gold_scorecard.agent_name IS NOT NULL AND gold_scorecard.empathy_score <= 2
GROUP BY gold_scorecard.agent_name

Answer:
A total of **12 agents** have been identified as needing coaching on empathy based on their empathy scores. Some of these agents include:
- **Carlos Kim**
- **Robert Nguyen**
- **Jennifer Park**
- **Kevin Garcia**
- **James Williams**
All agents listed have empathy scores of 2 or lower, indicating a need for improvement in this area.


## Presentation Guide (5-7 minutes)

### Narrative Arc:

**1. The Problem (30 sec)**
> "Our contact center handles X calls/day. QA supervisors manually review calls — it takes 45 min each. They can only review 2–3% of calls. Quality issues go undetected."

**2. What We Built (90 sec) — Live Demo**
> Run notebook 02 live. Show: raw transcript → AI scores in seconds.
> *"In 2 seconds, we get the same quality evaluation that takes a human 45 minutes."*

**3. We Don't Blindly Trust AI (60 sec)**
> Show notebook 03 results. 
> *"An independent LLM judge validates every score. Disagreements go to human review."*
> *"X% agreement within 1 point — matching human inter-rater reliability."*

**4. Supervisors Use This (60 sec) — Live Demo**
> Open the dashboard. Show agent rankings, outlier calls, coaching recommendations.
> *"Supervisors see actionable insights without writing any code."*

**5. Anyone Can Ask Questions (60 sec) — Live Demo**
> Open Genie Space. Ask: "Which agents need coaching on empathy?"
> *"Business analysts explore data in plain English. No SQL required."*

**6. Reusable + Scalable (30 sec)**
> *"The sentiment AI Skill is a reusable function — patient feedback team can use it tomorrow."*
> *"Low-confidence calls route to a HITL queue — next step is a Databricks App for supervisor review."*

---

### Rubric Score Targets:
| Criterion | Target | Evidence |
|---|---|---|
| 1. E2E Functionality (20%) | 5 | Live demo runs with no hand-holding |
| 2. Business Impact (20%) | 5 | "98% reduction in QA review time" + clear production path |
| 3. AI Quality (20%) | 4-5 | Agreement metrics + edge case handling |
| 4. Safety/Trust (15%) | 5 | LLM judge + HITL triage + confidence scoring |
| 5. UX (15%) | 5 | Dashboard + Genie Space (NL querying) |
| 6. Innovation (10%) | 5 | Reusable AI Skill UC function |

## Expansion: Human-in-the-Loop on Databricks

**What we built today (Day 1):**
```
Genesys → Stitch → AI Score → Judge Validate → Dashboard + Genie
                                     │
                              ┌──────┴────────┐
                              │ Disagreement? │
                              └──────┬────────┘
                                     │
                              HITL Triage Queue ← (table, done!)
```

**Production expansion (Week 2+):**
```
                              HITL Triage Queue
                                     │
                        ┌────────────┴───────────────┐
                        │ Databricks App (HITL UI)   │
                        │ Supervisor reviews call    │
                        │ Approves / Corrects score  │
                        └─────────────┬──────────────┘
                                      │
                        ┌─────────────┴─────────────┐
                        │ Corrections → Golden Set  │
                        │ Recalibrate judge prompt  │
                        │ Track drift over time     │
                        └───────────────────────────┘
```

**Databricks features for HITL at scale:**
* **Databricks Apps** — Custom UI for supervisor review workflow
* **Delta tables** — Audit trail of all human corrections
* **Lakeflow Jobs** — Scheduled recalibration of judge prompts
* **Alerts** — Notify when agreement drops below threshold
* **MLflow** — Track judge prompt versions and accuracy over time